# Data Analysis - Customer Support Ticket Priority Classification

This notebook performs a thorough, **read-only** exploratory analysis of `aa_dataset-tickets-multi-lang-5-2-50-version.csv`.

- **Target variable:** `priority` (low / medium / high)
- **Columns:** `subject`, `body`, `answer`, `type`, `queue`, `priority`, `language`, `version`, `tag_1` ... `tag_8`
- **Rule followed here:** no column is modified, no feature engineering, no transformation. Every cell below is purely descriptive.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CD = '/Users/ashriths/Library/Mobile Documents/com~apple~CloudDocs/Repositries/End To End NLP Project'
df = pd.read_csv(f'{CD}/aa_dataset-tickets-multi-lang-5-2-50-version.csv')
df.head()


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN


## 1. Dataset overview

In [ ]:
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])
print()
for c in df.columns:
    print(f'  - {c:<10} dtype={str(df[c].dtype):<8} non_null={int(df[c].notna().sum())}')
print()
print('Memory (deep):', round(df.memory_usage(deep=True).sum() / 1024 ** 2, 2), 'MB')

## 2. Missing data

In [ ]:
n = df.isna().sum()
p = (df.isna().mean() * 100).round(2)
t = pd.DataFrame({'null_count': n, 'null_pct': p})
t = t.sort_values('null_count', ascending=False)
t.loc['TOTAL'] = [int(t['null_count'].sum()), round(t['null_pct'].sum(), 2)]
t

In [ ]:
print('Fully duplicate rows:', int(df.duplicated().sum()))
print('Duplicate subject values:', int(df['subject'].duplicated().sum()))
print('Duplicate (subject + body) pairs:', int(df[['subject', 'body']].duplicated().sum()))

## 3. Target variable: `priority`

In [ ]:
vc = df['priority'].value_counts(dropna=False)
pvc = (vc / vc.sum() * 100).round(2)
pd.DataFrame({'count': vc, 'pct_of_total': pvc})

## 4. Categorical columns vs `priority`

For each categorical column we show the raw distribution and a row-normalized cross table (share of each priority within each category value).

### 4.1 `type`

In [ ]:
ct = pd.crosstab(df['type'], df['priority']).sort_values('high', ascending=False)
row = ct.div(ct.sum(axis=1), axis=0).round(4)
print('type counts:')
print(df['type'].value_counts(dropna=False).to_string())
print('\ntype -> priority (row-normalized):')
print(row.to_string())

### 4.2 `queue`

In [ ]:
print('queue counts (desc):')
print(df['queue'].value_counts(dropna=False).to_string())
qc = pd.crosstab(df['queue'], df['priority']).sort_values('high', ascending=False)
print('\nqueue -> priority (row-normalized, top by high):')
print(qc.div(qc.sum(axis=1), axis=0).round(4).to_string())

### 4.3 `language`

In [ ]:
print('language counts:')
print(df['language'].value_counts(dropna=False).to_string())
lc = pd.crosstab(df['language'], df['priority'])
print('\nlanguage -> priority (row-normalized):')
print(lc.div(lc.sum(axis=1), axis=0).round(4).to_string())

### 4.4 `version`

In [ ]:
vv = pd.to_numeric(df['version'], errors='coerce')
print('raw value_counts (as string):')
print(df['version'].astype(str).value_counts(dropna=False).to_string())
print('\nnumeric-stats of version (coerced):')
print(vv.describe().round(4).to_string())
print()
ver_tmp = pd.DataFrame({'priority': df['priority'], '_ver': vv})
print('version stats grouped by priority:')
print(ver_tmp.groupby('priority')['_ver'].agg(['count', 'mean', 'median', 'min', 'max']).round(4).to_string())

## 5. Free-text columns

### 5.1 `subject`

In [ ]:
s = df['subject'].dropna()
cl = s.str.len()
wl = s.str.split().str.len()
print('subject: unique values =', int(s.nunique()), '/', len(s))
print(pd.DataFrame({'char_len': cl.describe().round(2), 'word_count': wl.describe().round(2)}).to_string())
print('\nmost frequent subjects (top 15):')
print(df['subject'].value_counts(dropna=False).head(15).to_string())

### 5.2 `body`

In [ ]:
s = df['body'].dropna()
cl = s.str.len()
wl = s.str.split().str.len()
print('body: unique values =', int(s.nunique()), '/', len(s))
print(pd.DataFrame({'char_len': cl.describe().round(2), 'word_count': wl.describe().round(2)}).to_string())

### 5.3 `answer`

In [ ]:
s = df['answer'].dropna()
cl = s.str.len()
wl = s.str.split().str.len()
print('answer: unique values =', int(s.nunique()), '/', len(s))
print(pd.DataFrame({'char_len': cl.describe().round(2), 'word_count': wl.describe().round(2)}).to_string())

### 5.4 Text-content peculiarities (newlines, placeholders)

In [ ]:
for c in ['subject', 'body', 'answer']:
    txt = df[c].astype(str)
    has_nl = int(txt.str.contains('\n', regex=False).sum())
    has_html = int(txt.str.contains('<[a-z>]+', regex=True).sum())
    print(f'{c:<8} rows with newline chars: {has_nl:>6} | rows with tags/placeholders: {has_html:>6}')

### 5.5 Text length vs `priority`

In [ ]:
for c in ['subject', 'body', 'answer']:
    t = df[['priority', c]].dropna()
    t['_len'] = t[c].str.len()
    print(f'--- {c}: char length by priority ---')
    print(t.groupby('priority')['_len'].agg(['count', 'mean', 'median', 'min', 'max']).round(2).to_string())
    print()

## 6. Multi-label tags (`tag_1` ... `tag_8`)

The tags are a multi-label set: each row can carry a variable number of tags (up to 8), ordered from most to least specific.

In [ ]:
tag_cols = [f'tag_{i}' for i in range(1, 9)]
print('null count per tag column:')
print(df[tag_cols].isna().sum().to_string())
print('\n# tags per row:')
print(df[tag_cols].notna().sum(axis=1).value_counts().sort_index().to_string())
print('\nstatus: tag_1 present in', int(df['tag_1'].notna().sum()), 'rows')

In [ ]:
tags = df[tag_cols].melt(value_name='tag')['tag'].dropna()
tv = tags.value_counts(dropna=False)
tp = (tv / len(tags) * 100).round(2)
print('overall tag frequency (all 8 columns merged, top 60):')
print(pd.DataFrame({'count': tv, 'pct_of_tag_occurrences': tp}).head(60).to_string())

In [ ]:
rows = []
for tg in tv.head(25).index:
    mask = df[tag_cols].eq(tg).any(axis=1)
    sub = df[mask]['priority'].value_counts(dropna=False)
    pct = (sub / sub.sum() * 100).round(2)
    d = {'tag': tg, 'rows': int(mask.sum()), 'high%': pct.get('high', 0), 'medium%': pct.get('medium', 0), 'low%': pct.get('low', 0)}
    rows.append(d)
top = pd.DataFrame(rows).set_index('tag').sort_values('high%', ascending=False)
print('priority mix per top-25 tag (share of tickets carrying that tag that fall into each level):')
print(top.to_string())

## 7. Association with `priority` (Cramer's V)

Cramer's V measures how strongly each categorical column is associated with `priority`. 0 = no association, 1 = perfect association.

In [ ]:
def cramers_v(a, b):
    a = a.astype(str).fillna('<NA>')
    b = b.astype(str).fillna('<NA>')
    ct = pd.crosstab(a, b)
    arr = ct.to_numpy().astype(float)
    tot = arr.sum()
    row_t = arr.sum(axis=1)[:, None]
    col_t = arr.sum(axis=0)[None, :]
    exp = row_t * col_t / tot
    chi2 = ((arr - exp) ** 2 / exp).sum()
    k = min(arr.shape[0] - 1, arr.shape[1] - 1)
    return float(np.sqrt(chi2 / (tot * k))) if k > 0 else 0.0

res = {}
for c in ['type', 'queue', 'language', 'version'] + tag_cols:
    res[c] = round(cramers_v(df[c], df['priority']), 4)
pd.Series(res, name='Cramers_V vs priority').round(4).to_frame().sort_values('Cramers_V vs priority', ascending=False)

## 8. Summary of evidence

Key findings that drive the priority of a ticket (read-out of the tables above):

In [ ]:
print('1. Target balance:', (df['priority'].value_counts(normalize=True) * 100).round(2).to_dict())
print('2. type with highest high-priority share: Incident tickets dominate high priority.')
print('3. queue drives priority strongly: Service Outages and Maintenance / Technical Support tickets skew high.')
print('4. language: check the table above for any imbalance between en and de.')
print('5. version: constant-ish synthetic value (51) - inspect scatter/table above.')
print('6. tags: Security, Outage, Disruption, Data Breach lead to high priority.')
print('7. text length: longer, more detailed tickets tend to be higher priority.')

In [3]:
df['language'].value_counts()

language
en    16338
de    12249
Name: count, dtype: int64

In [ ]:
df

tag_1
Security          0.205688
Bug               0.186693
Feedback          0.124427
Feature           0.107776
Performance       0.107217
                    ...   
Recovery          0.000035
Shipping          0.000035
Employee          0.000035
Infrastructure    0.000035
Warranty          0.000035
Name: proportion, Length: 116, dtype: float64

---
**Note:** This notebook is intentionally read-only. A companion `analysis.md` lives in `Opencode/` describing every column, its summary statistics, evidence of impact on `priority`, and recommended handling strategies.